In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import AutoModel
from huggingface_hub import login, notebook_login
import ast
from datasets import Dataset
from tqdm.auto import tqdm
from google.colab import drive
import os
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
drive.mount('/content/drive')
login(token="token")

In [ ]:
notebook_login()

In [ ]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at',
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords',
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

!cp '/content/drive/My Drive/GDS_group_project_main/Group_project/test_set.csv' '/content/local_dataset.csv'

test_set = pd.read_csv(
    '/content/local_dataset.csv',
    converters = converters_dict)

In [ ]:
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'reliabl', 'unreli', 'polit', 'clickbait'}

def filter_type(type, fake, rel):
    if type in fake:
        return 0
    elif type in rel:
        return 1

In [ ]:
model_id = 'model'
hf_token = 'token'
tokenizer = AutoTokenizer.from_pretrained(model_id, hf_token)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_ds(ds):
    return tokenizer(
        ds['content'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [ ]:
test_data = pd.DataFrame({})
test_data['content'] = test_set['content'].apply(lambda x, **kwargs: ' '.join(x))
test_data['type'] = test_set['type'].apply(lambda x, **kwargs: filter_type(' '.join(x), fake_news, reliable_news))
filtered_test = test_data[test_data['type'].apply(lambda x, **kwargs: x in {0, 1})]

In [ ]:
cores = max(1, os.cpu_count() - 1)
hf_test = Dataset.from_pandas(filtered_test)
test_tokenized = hf_test.map(tokenize_ds, batched=True, batch_size=1000, num_proc=cores)
test_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'type'])

test_loader = DataLoader(
    test_tokenized,
    batch_size=256,
    shuffle=False,
    num_workers=cores,
    pin_memory=True
)

In [ ]:
class llama_text_classifier(nn.Module):
    def __init__(self, model_id, token, num_classes, hidden_size=4096):
        super(llama_text_classifier, self).__init__()

        self.llama = AutoModel.from_pretrained(
            model_id, token = hf_token, torch_dtype=torch.bfloat16)

        for param in self.llama.parameters():
            param.requires_grad = False

        self.custom_layers = nn.Sequential(
        nn.Linear(hidden_size, 1024),
        nn.Sigmoid(),
        nn.Dropout(0.2),
        nn.Linear(1024, 256),
        nn.Sigmoid(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes))

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.llama(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state

            input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()

            sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

            mean_pooled_states = sum_embeddings / sum_mask
            mean_pooled_states = mean_pooled_states.to(torch.float32)

        logits = self.custom_layers(mean_pooled_states)
        return logits

In [ ]:
num_classes = len(filtered_test['type'].unique())

model = llama_text_classifier(model_id="model", token=hf_token, num_classes=num_classes)

save_path = '/content/drive/My Drive/GDS_group_project_main/Group_project/custom_weights_tore.pth'
model.custom_layers.load_state_dict(torch.load(save_path))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
model.eval()

all_pred = []
all_true = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['type'].to(device, dtype=torch.long)

        logits = model(input_ids, attention_mask)

        predictions = torch.argmax(logits, dim=-1)

        all_pred.extend(predictions.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

In [ ]:
print('The confusion matrix on the test set')
print(confusion_matrix(all_true, all_pred))
print('The classification report on the test set')
print(classification_report(all_true, all_pred))